# Chapter 6 — Agent Skills

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

<!-- provider-note:start -->
## Setting up the LLM

I recommend running these notebooks on **Ollama Cloud**. It serves much
larger models than most laptops can run. The default cloud model is picked
by benchmarking candidates on these notebooks, and an automated run checks
every notebook against it each week. The
[notebook status page](https://masfromscratch.com/notebook-status/) shows
the current model and results.

1. Create an API key at https://ollama.com/settings/keys.
2. Export it before launching Jupyter: `export OLLAMA_API_KEY=<your key>`.
3. That's all. When a notebook builds its LLM, it detects the key and uses
   Ollama Cloud.

**Prefer to run locally?** Install Ollama from https://ollama.com/download
and pull the model with `ollama pull qwen3:14b`. If no key is set, the
notebook starts the Ollama service for you, so there is no need to run
`ollama serve`. Local models are smaller, so your outputs will differ from
the recorded ones.

**Prefer OpenAI or Anthropic?**

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
   If you forget, you will be prompted for it, which keeps the key out of
   the saved notebook.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama. Output on
gpt-5 or Claude will differ from the recorded outputs, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

In [2]:
from llm_agents_from_scratch.notebook_utils import make_llm

## The Hailstone Skill

The examples in this notebook use a `hailstone-sequence` skill that instructs the agent to compute a full Hailstone (Collatz) sequence using a `next_number` tool.

**Important:** Run this notebook from within the `chapter-notebooks/ch06/` directory so the skill is discovered as a project-scoped skill.

The skill is located at: `chapter-notebooks/shared/.agents/skills/stop-at-one/SKILL.md`, linked into this chapter's folder as `.agents`

> **Note for Windows:** `.agents` in this folder is a symlink to `chapter-notebooks/shared/.agents`. Git only checks symlinks out as real links when Developer Mode or `git config core.symlinks true` is enabled; otherwise `.agents` is a small text file and the agent won't find the `stop-at-one` skill. In that case, copy `chapter-notebooks/shared/.agents` into this folder.

> **Note on obfuscation:** The skill is intentionally named `stop-at-one` and contains no mention of "Hailstone" or "Collatz". This prevents the LLM from relying on its parametric knowledge of the Collatz conjecture to simulate the sequence internally — forcing it to actually call the `next_number` tool at each step as the skill instructs.

## Examples

### Example 1: Skill Discovery

In [3]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task

llm = make_llm()
agent = LLMAgent(llm=llm)

task = Task(instruction="Compute the stop-at-one sequence for 8.")
handler = LLMAgent.TaskHandler(llm_agent=agent, task=task)

print(handler.skills_registry)
print(handler.skills_registry["stop-at-one"].frontmatter)
print(handler.skills_registry["stop-at-one"].scope)

✓ Using Ollama Cloud (deepseek-v4.1-flash)
{'stop-at-one': <llm_agents_from_scratch.skills.skill.Skill object at 0x7e9e4d6216a0>}
name='stop-at-one' description='Compute a sequence for a given starting number by repeatedly calling the next_number tool until the value 1 is reached.' license=None compatibility=None metadata=None allowed_tools=None
SkillScope.PROJECT


### Example 2: UseSkillTool Activation Result

In [4]:
from llm_agents_from_scratch.data_structures import ToolCall
from llm_agents_from_scratch.skills.tools import UseSkillTool

tool = UseSkillTool(skills_registry=handler.skills_registry)
tool_call = ToolCall(
    tool_name="from_scratch__use_skill",
    arguments={"name": "stop-at-one"},
)
result = tool(tool_call=tool_call)
print(result.content)

<skill_content name="stop-at-one">
# Stop At One

Compute a full sequence from a starting number down to 1
using the `next_number` tool.

## Arguments

The user must provide a **starting number** (a positive integer greater than 1).

If no starting number is given, ask the user before proceeding.

## Steps

### 1. Initialize

Set the current number `x` to the starting number provided by the user.

Begin tracking the sequence as a list: `[x]`.

### 2. Call the tool

Call `next_number` with the current value of `x`.

```
next_number(x=<current_number>)
```

**STOP and WAIT** for the tool result before continuing.

### 3. Record the result

Append the returned value to the sequence list.

Set `x` to the returned value.

### 4. Check termination

- If `x == 1`, proceed to Step 5.
- Otherwise, go back to Step 2.

**Important rules:**
- NEVER fabricate or simulate tool call results.
- NEVER make multiple tool calls in a single response.
- ALWAYS wait for the actual tool response before decid

### Example 3: Model-driven Skill Activation

In [5]:
import logging
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

In [6]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)
llm = make_llm()
agent = LLMAgent(llm=llm, tools=[next_number_tool])

task = Task(instruction="Compute the stop-at-one sequence for 8.")
result = await agent.run(task)

✓ Using Ollama Cloud (deepseek-v4.1-flash)
INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the stop-at-one sequence for 8.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the stop-at-one sequence for 8.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_skill
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: <skill_content name="stop-at-one">
# Stop At One

Compute a full sequence from a starting number down to 1
using the `next_num...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "a245e16a-5f2f-4f66-8496-d0296b5e7dc1",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the pending tool call: next_number with x=8, then wait for the result before continuing the stop-at-one sequence.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Execute the pending tool call: 

In [7]:
print(result)

The sequence has reached 1, so it's complete. Let me report the full result.

**Stop-at-one sequence for 8:**

```
8 → 4 → 2 → 1
```

- **Starting number:** 8
- **Total steps taken:** 3 tool calls (8→4, 4→2, 2→1)
- **Maximum value reached:** 8
